# Bilingual Indian Financial Document Extractor
### Fine-tuning Qwen2.5-VL-3B for Structured Data Extraction from Hindi-Gujarati-English Bank Statements

---

**Author:** Priya Patel  
**Date:** May 2026  
**Model:** Qwen2.5-VL-3B-Instruct + LoRA (via Unsloth)  
**Framework:** Unsloth + TRL SFTTrainer  

---

## Abstract

Indian financial documents — bank statements, insurance policies, mutual fund reports — are predominantly **bilingual or trilingual**, mixing English with Hindi (Devanagari) and regional scripts like Gujarati. Global document AI models treat these languages as secondary, leading to significant extraction errors on Indian documents.

This notebook fine-tunes **Qwen2.5-VL-3B** using **Unsloth + LoRA** on a custom dataset of **1,300 synthetic Indian bank statements** (2,913 page-level samples) spanning real and fictional Indian bank formats with bilingual narrations. We evaluate structured JSON extraction accuracy **before and after fine-tuning**, measuring:

- **Field-level exact match** (account number, IFSC, balances, etc.)
- **Transaction extraction accuracy** (date, amount, description matching)
- **Indic script handling** (Hindi and Gujarati description preservation)
- **Numerical precision** (₹ amounts, running balances)

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Dataset Loading & Exploration](#2-dataset-loading--exploration)
3. [Baseline Evaluation (Pre-Fine-Tuning)](#3-baseline-evaluation)
4. [Fine-Tuning with LoRA](#4-fine-tuning-with-lora)
5. [Post-Fine-Tuning Evaluation](#5-post-fine-tuning-evaluation)
6. [Results Comparison & Analysis](#6-results-comparison--analysis)
7. [Qualitative Examples](#7-qualitative-examples)
8. [Export & Deployment](#8-export--deployment)

---
## 1. Environment Setup <a id='1-environment-setup'></a>

We use **Unsloth** for optimized VLM fine-tuning — same library used in our Gemma 4 experiments.  
Most dependencies (torch, PIL, matplotlib, numpy, pandas) are pre-installed in Colab.  
We only install the extras we need.

In [ ]:
# ── 1.1 Install Only What's Missing ──────────────────────────────────────
%%capture
!pip install unsloth
!pip install rapidfuzz editdistance tabulate

In [ ]:
# ── 1.2 GPU Detection & Adaptive Config ──────────────────────────────────
import torch

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
compute_cap = torch.cuda.get_device_capability()

print(f"GPU:     {gpu_name}")
print(f"VRAM:    {vram_gb:.1f} GB")
print(f"Compute: {compute_cap}")
print(f"PyTorch: {torch.__version__}")

# Adaptive configuration based on available GPU
if vram_gb >= 70:
    # H100 / A100 80GB
    GPU_TIER = "high"
    LOAD_4BIT = False
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
    LORA_R = 128
    NUM_EPOCHS = 3
    print("\nHigh-VRAM GPU detected. 16-bit LoRA, batch=4, r=128.")
elif vram_gb >= 30:
    # A100 40GB / L4
    GPU_TIER = "mid"
    LOAD_4BIT = False
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    LORA_R = 64
    NUM_EPOCHS = 3
    print("\nMid-VRAM GPU detected. 16-bit LoRA, batch=2, r=64.")
elif vram_gb >= 20:
    # L4 24GB
    GPU_TIER = "mid-low"
    LOAD_4BIT = True
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    LORA_R = 64
    NUM_EPOCHS = 3
    print("\nL4-class GPU detected. 4-bit QLoRA, batch=2, r=64.")
else:
    # T4 16GB
    GPU_TIER = "low"
    LOAD_4BIT = True
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    LORA_R = 32
    NUM_EPOCHS = 3
    print("\nT4-class GPU detected. 4-bit QLoRA, batch=1, r=32.")

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
USE_BF16 = compute_cap[0] >= 8  # bf16 on Ampere+
print(f"Effective batch: {EFFECTIVE_BATCH}")
print(f"Precision: {'bf16' if USE_BF16 else 'fp16'}")

In [ ]:
# ── 1.3 Imports ─────────────────────────────────────────────────────────
import json
import os
import re
import time
import warnings
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tabulate import tabulate
from rapidfuzz import fuzz
from tqdm import tqdm

from unsloth import FastVisionModel

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

print("All imports successful.")

In [ ]:
# ── 1.4 Configuration ──────────────────────────────────────────────────────
CONFIG = {
    # Model
    "model_id": "unsloth/Qwen2.5-VL-3B-Instruct",
    "max_seq_length": 4096,
    
    # Dataset paths
    "dataset_dir": "vlm_training_dataset",
    "train_file": "vlm_training_dataset/train.jsonl",
    "val_file": "vlm_training_dataset/val.jsonl",
    "test_file": "vlm_training_dataset/test.jsonl",
    
    # LoRA (set by GPU detection)
    "lora_r": LORA_R,
    "lora_alpha": LORA_R,       # alpha = r (standard practice)
    "lora_dropout": 0,
    
    # Training (set by GPU detection)
    "load_4bit": LOAD_4BIT,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": 5e-5,
    "warmup_ratio": 0.05,
    "bf16": USE_BF16,
    "fp16": not USE_BF16,
    
    # Evaluation
    "eval_samples": 20,
    "max_new_tokens": 4096,
    
    # Output
    "output_dir": "./qwen_vlm_finetuned",
    "results_dir": "./results",
}

os.makedirs(CONFIG["results_dir"], exist_ok=True)
print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

---
## 2. Dataset Loading & Exploration <a id='2-dataset-loading--exploration'></a>

In [ ]:
# ── 2.1 Upload Dataset ─────────────────────────────────────────────────────
# Upload vlm_training_dataset.zip to Colab, or mount Google Drive.

# --- Option A: Upload zip ---
# from google.colab import files
# uploaded = files.upload()  # upload vlm_training_dataset.zip
# !unzip -q vlm_training_dataset.zip

# --- Option B: Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/vlm_training_dataset ./

print("Dataset directory:")
!ls -la {CONFIG['dataset_dir']}/
print(f"\nImages: {len(list(Path(CONFIG['dataset_dir'], 'images').glob('*.png')))}")

In [ ]:
# ── 2.2 Load JSONL Splits ──────────────────────────────────────────────────
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(CONFIG["train_file"])
val_data = load_jsonl(CONFIG["val_file"])
test_data = load_jsonl(CONFIG["test_file"])

print(f"Train: {len(train_data)} samples")
print(f"Val:   {len(val_data)} samples")
print(f"Test:  {len(test_data)} samples")
print(f"Total: {len(train_data) + len(val_data) + len(test_data)} samples")

In [ ]:
# ── 2.3 Dataset Statistics ─────────────────────────────────────────────────
def analyze_split(data, name):
    sources = defaultdict(int)
    txn_counts = []
    resp_lengths = []
    
    for s in data:
        src = "DS1 (Bilingual)" if s["source"].startswith("ds1") else "DS2 (Synthetic)"
        sources[src] += 1
        resp = json.loads(s["conversations"][2]["content"])
        txn_counts.append(len(resp.get("transactions", [])))
        resp_lengths.append(len(s["conversations"][2]["content"]))
    
    print(f"\n{'='*50}")
    print(f"  {name} Split Analysis")
    print(f"{'='*50}")
    print(f"  Total samples: {len(data)}")
    print(f"  Source distribution: {dict(sources)}")
    print(f"  Avg transactions/sample: {np.mean(txn_counts):.1f}")
    print(f"  Avg response length: {np.mean(resp_lengths):.0f} chars")
    print(f"  Max response length: {max(resp_lengths)} chars")

analyze_split(train_data, "Train")
analyze_split(val_data, "Validation")
analyze_split(test_data, "Test")

In [ ]:
# ── 2.4 Visualize Sample Documents ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 16))
fig.suptitle("Sample Bank Statement Pages from Dataset", fontsize=16, fontweight="bold")

sample_indices = []
for i, s in enumerate(train_data):
    if s["source"].startswith("ds1") and s["page"] == 1 and len(sample_indices) < 3:
        sample_indices.append(i)
    if s["source"].startswith("ds2") and s["page"] == 1 and len(sample_indices) < 6 and len(sample_indices) >= 3:
        sample_indices.append(i)

for idx, ax in zip(sample_indices[:6], axes.flat):
    sample = train_data[idx]
    img = Image.open(Path(CONFIG["dataset_dir"]) / sample["image"])
    ax.imshow(img)
    ax.set_title(f"{sample['id']}\n({sample['page']}/{sample['total_pages']} pages)", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/sample_documents.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2.5 Inspect One Sample ─────────────────────────────────────────────────
sample = train_data[0]
print(f"Sample ID: {sample['id']}")
print(f"Image:     {sample['image']}")
print(f"Page:      {sample['page']} / {sample['total_pages']}")
print(f"\n--- System Prompt ---")
print(sample["conversations"][0]["content"][:200])
print(f"\n--- User Prompt ---")
print(sample["conversations"][1]["content"][:300])
print(f"\n--- Expected JSON (first 500 chars) ---")
print(sample["conversations"][2]["content"][:500])

---
## 3. Baseline Evaluation (Pre-Fine-Tuning) <a id='3-baseline-evaluation'></a>

Before fine-tuning, we evaluate the **base Qwen2.5-VL-3B-Instruct** on our test set.  
This establishes the performance floor and reveals where the base model fails on Indian financial documents.

In [ ]:
# ── 3.1 Load Base Model via Unsloth ───────────────────────────────────────
print(f"Loading base model ({'4-bit' if CONFIG['load_4bit'] else '16-bit'})...")

model, processor = FastVisionModel.from_pretrained(
    CONFIG["model_id"],
    load_in_4bit=CONFIG["load_4bit"],
    use_gradient_checkpointing="unsloth",
    max_seq_length=CONFIG["max_seq_length"],
)

print(f"Model loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── 3.2 Evaluation Metrics ─────────────────────────────────────────────────

def parse_json_response(text):
    """Parse JSON from model output, handling markdown blocks and trailing text."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    for pattern in [r'```json\s*\n(.*?)\n```', r'```\s*\n(.*?)\n```', r'\{[\s\S]*\}']:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1) if '```' in pattern else match.group())
            except (json.JSONDecodeError, IndexError):
                continue
    return None


def compute_field_accuracy(predicted, ground_truth, field_name):
    """Compare a single field. Returns (score 0-1, match_type)."""
    gt_val = ground_truth.get(field_name)
    pred_val = predicted.get(field_name) if predicted else None
    
    if gt_val is None and pred_val is None:
        return 1.0, "both_null"
    if gt_val is None or pred_val is None:
        return 0.0, "missing"
    
    # Numeric
    if isinstance(gt_val, (int, float)) and isinstance(pred_val, (int, float)):
        if abs(gt_val - pred_val) < 0.01:
            return 1.0, "exact_numeric"
        elif abs(gt_val - pred_val) / max(abs(gt_val), 1) < 0.01:
            return 0.8, "close_numeric"
        return 0.0, "wrong_numeric"
    
    # String
    gt_str, pred_str = str(gt_val).strip(), str(pred_val).strip()
    if gt_str == pred_str:
        return 1.0, "exact_string"
    if gt_str.lower() == pred_str.lower():
        return 0.95, "case_insensitive"
    ratio = fuzz.ratio(gt_str.lower(), pred_str.lower()) / 100.0
    if ratio > 0.85:
        return ratio, "fuzzy"
    return 0.0, "mismatch"


def compute_transaction_metrics(pred_txns, gt_txns):
    """Compare transaction lists. Returns per-field accuracy."""
    if not pred_txns or not gt_txns:
        return {"count_accuracy": 1.0 if len(pred_txns or []) == len(gt_txns or []) else 0.0,
                "date_accuracy": 0.0, "amount_accuracy": 0.0,
                "description_similarity": 0.0, "balance_accuracy": 0.0}
    
    n = min(len(pred_txns), len(gt_txns))
    date_s, amt_s, desc_s, bal_s = [], [], [], []
    
    for i in range(n):
        pt = pred_txns[i] if isinstance(pred_txns[i], dict) else {}
        gt = gt_txns[i]
        
        s, _ = compute_field_accuracy(pt, gt, "date")
        date_s.append(s)
        
        d_s, _ = compute_field_accuracy(pt, gt, "debit")
        c_s, _ = compute_field_accuracy(pt, gt, "credit")
        amt_s.append((d_s + c_s) / 2)
        
        desc_s.append(fuzz.ratio(str(gt.get("description", "")),
                                 str(pt.get("description", ""))) / 100.0)
        s, _ = compute_field_accuracy(pt, gt, "balance")
        bal_s.append(s)
    
    return {"count_accuracy": n / max(len(gt_txns), 1),
            "date_accuracy": np.mean(date_s), "amount_accuracy": np.mean(amt_s),
            "description_similarity": np.mean(desc_s), "balance_accuracy": np.mean(bal_s)}


HEADER_FIELDS = ["bank_name", "account_holder_name", "account_number",
                 "ifsc", "account_type", "opening_balance", "closing_balance"]

def evaluate_sample(pred_json, gt_json):
    """Full evaluation of one sample."""
    results = {"json_parseable": pred_json is not None}
    
    if pred_json is None:
        for f in HEADER_FIELDS:
            results[f"field_{f}"] = 0.0
        results.update({"header_avg": 0.0, "txn_count_accuracy": 0.0,
                        "txn_date_accuracy": 0.0, "txn_amount_accuracy": 0.0,
                        "txn_desc_similarity": 0.0, "txn_balance_accuracy": 0.0})
        return results
    
    header_scores = []
    for f in HEADER_FIELDS:
        s, _ = compute_field_accuracy(pred_json, gt_json, f)
        results[f"field_{f}"] = s
        if gt_json.get(f) is not None:
            header_scores.append(s)
    results["header_avg"] = np.mean(header_scores) if header_scores else 0.0
    
    txn = compute_transaction_metrics(pred_json.get("transactions", []),
                                      gt_json.get("transactions", []))
    results["txn_count_accuracy"] = txn["count_accuracy"]
    results["txn_date_accuracy"] = txn["date_accuracy"]
    results["txn_amount_accuracy"] = txn["amount_accuracy"]
    results["txn_desc_similarity"] = txn["description_similarity"]
    results["txn_balance_accuracy"] = txn["balance_accuracy"]
    return results

print("Evaluation metrics defined.")

In [ ]:
# ── 3.3 Inference Function (Unsloth) ──────────────────────────────────────
def run_inference(model, processor, image_path, system_prompt, user_prompt,
                  max_new_tokens=4096):
    """
    Run inference on a single image using Unsloth's optimized path.
    """
    image = Image.open(image_path).convert("RGB")
    
    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_prompt.replace("<image>\n", "")},
        ]},
    ]
    
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    
    start_time = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.time() - start_time
    
    # Decode only generated tokens
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    output_text = processor.decode(gen_ids, skip_special_tokens=True).strip()
    
    return output_text, elapsed

print("Inference function defined.")

In [ ]:
# ── 3.4 Run Baseline Evaluation ───────────────────────────────────────────
def run_evaluation(model, processor, test_data, tag, n_samples=None):
    """Run evaluation on test set and return detailed results."""
    eval_data = test_data[:n_samples] if n_samples else test_data
    
    all_results = []
    raw_predictions = []
    
    print(f"\n{'='*60}")
    print(f"  Running {tag} Evaluation ({len(eval_data)} samples)")
    print(f"{'='*60}")
    
    for i, sample in enumerate(eval_data):
        img_path = Path(CONFIG["dataset_dir"]) / sample["image"]
        system_prompt = sample["conversations"][0]["content"]
        user_prompt = sample["conversations"][1]["content"]
        gt_json = json.loads(sample["conversations"][2]["content"])
        
        print(f"  [{i+1}/{len(eval_data)}] {sample['id']}...", end=" ")
        
        try:
            output_text, elapsed = run_inference(
                model, processor, img_path, system_prompt, user_prompt,
                max_new_tokens=CONFIG["max_new_tokens"],
            )
            pred_json = parse_json_response(output_text)
            metrics = evaluate_sample(pred_json, gt_json)
            metrics["inference_time"] = elapsed
            metrics["sample_id"] = sample["id"]
            metrics["source"] = "bilingual" if sample["source"].startswith("ds1") else "synthetic"
            
            raw_predictions.append({
                "sample_id": sample["id"],
                "prediction": output_text[:2000],
                "parsed": pred_json is not None,
            })
            
            status = "OK" if pred_json else "PARSE_FAIL"
            print(f"{status} ({elapsed:.1f}s, header={metrics['header_avg']:.2f})")
            
        except Exception as e:
            print(f"ERROR: {e}")
            metrics = {"json_parseable": False, "header_avg": 0.0,
                       "txn_count_accuracy": 0.0, "txn_date_accuracy": 0.0,
                       "txn_amount_accuracy": 0.0, "txn_desc_similarity": 0.0,
                       "txn_balance_accuracy": 0.0, "inference_time": 0.0,
                       "sample_id": sample["id"],
                       "source": "bilingual" if sample["source"].startswith("ds1") else "synthetic"}
            for f in HEADER_FIELDS:
                metrics[f"field_{f}"] = 0.0
        
        all_results.append(metrics)
    
    with open(f"{CONFIG['results_dir']}/{tag}_predictions.json", "w", encoding="utf-8") as f:
        json.dump(raw_predictions, f, ensure_ascii=False, indent=2)
    
    return all_results


# Set model to inference mode
FastVisionModel.for_inference(model)

baseline_results = run_evaluation(
    model, processor, test_data,
    tag="baseline",
    n_samples=CONFIG["eval_samples"],
)

In [ ]:
# ── 3.5 Results Summary Function ──────────────────────────────────────────
def summarize_results(results, tag):
    """Print formatted summary and return metrics dict."""
    json_rate = np.mean([r["json_parseable"] for r in results]) * 100
    header_avg = np.mean([r["header_avg"] for r in results]) * 100
    
    field_scores = {}
    for f in HEADER_FIELDS:
        scores = [r[f"field_{f}"] for r in results if f"field_{f}" in r]
        field_scores[f] = np.mean(scores) * 100 if scores else 0.0
    
    txn_count = np.mean([r["txn_count_accuracy"] for r in results]) * 100
    txn_date = np.mean([r["txn_date_accuracy"] for r in results]) * 100
    txn_amount = np.mean([r["txn_amount_accuracy"] for r in results]) * 100
    txn_desc = np.mean([r["txn_desc_similarity"] for r in results]) * 100
    txn_bal = np.mean([r["txn_balance_accuracy"] for r in results]) * 100
    
    bilingual = [r for r in results if r.get("source") == "bilingual"]
    synthetic = [r for r in results if r.get("source") == "synthetic"]
    avg_time = np.mean([r["inference_time"] for r in results])
    
    print(f"\n{'='*60}")
    print(f"  {tag} RESULTS")
    print(f"{'='*60}")
    
    table = [
        ["JSON Parse Rate", f"{json_rate:.1f}%"],
        ["Header Fields (avg)", f"{header_avg:.1f}%"],
        ["", ""], ["── Header Fields ──", ""],
    ]
    for f, s in field_scores.items():
        table.append([f"  {f}", f"{s:.1f}%"])
    table.extend([
        ["", ""], ["── Transactions ──", ""],
        ["  Count accuracy", f"{txn_count:.1f}%"],
        ["  Date accuracy", f"{txn_date:.1f}%"],
        ["  Amount accuracy", f"{txn_amount:.1f}%"],
        ["  Description similarity", f"{txn_desc:.1f}%"],
        ["  Balance accuracy", f"{txn_bal:.1f}%"],
        ["", ""], ["── By Source ──", ""],
        ["  Bilingual (Hi+Gu+En)",
         f"{np.mean([r['header_avg'] for r in bilingual])*100:.1f}%" if bilingual else "N/A"],
        ["  Synthetic (En blend)",
         f"{np.mean([r['header_avg'] for r in synthetic])*100:.1f}%" if synthetic else "N/A"],
        ["", ""],
        ["Avg inference time", f"{avg_time:.1f}s"],
    ])
    print(tabulate(table, headers=["Metric", "Score"], tablefmt="simple"))
    
    return {
        "json_rate": json_rate, "header_avg": header_avg, "field_scores": field_scores,
        "txn_count": txn_count, "txn_date": txn_date, "txn_amount": txn_amount,
        "txn_desc": txn_desc, "txn_balance": txn_bal, "avg_time": avg_time,
        "bilingual_header": np.mean([r['header_avg'] for r in bilingual])*100 if bilingual else 0,
        "synthetic_header": np.mean([r['header_avg'] for r in synthetic])*100 if synthetic else 0,
    }


baseline_summary = summarize_results(baseline_results, "BASELINE (Pre-Fine-Tuning)")

In [ ]:
# ── 3.6 Save Baseline & Free Memory ──────────────────────────────────────
with open(f"{CONFIG['results_dir']}/baseline_summary.json", "w") as f:
    json.dump(baseline_summary, f, indent=2)
with open(f"{CONFIG['results_dir']}/baseline_detailed.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)

print("Baseline results saved.")

# Free memory before training
del model, processor
torch.cuda.empty_cache()
import gc; gc.collect()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## 4. Fine-Tuning with LoRA <a id='4-fine-tuning-with-lora'></a>

We fine-tune Qwen2.5-VL-3B using **Unsloth + LoRA** — the same framework used in our Gemma 4 experiments.  
Unsloth provides:
- **2x faster training** than vanilla HuggingFace
- **Optimized gradient checkpointing** — less VRAM, same speed
- **Rank-stabilized LoRA (rsLoRA)** — better convergence at high ranks
- **UnslothVisionDataCollator** — handles vision-language batching automatically

In [ ]:
# ── 4.1 Reload Model for Training ─────────────────────────────────────────
print(f"Loading model for fine-tuning ({'4-bit' if CONFIG['load_4bit'] else '16-bit'})...")

model, processor = FastVisionModel.from_pretrained(
    CONFIG["model_id"],
    load_in_4bit=CONFIG["load_4bit"],
    use_gradient_checkpointing="unsloth",
    max_seq_length=CONFIG["max_seq_length"],
)

print(f"Loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── 4.2 Apply LoRA Adapters ───────────────────────────────────────────────
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = CONFIG["lora_r"],
    lora_alpha = CONFIG["lora_alpha"],
    lora_dropout = CONFIG["lora_dropout"],
    bias = "none",
    random_state = 3407,
    use_rslora = True,             # Rank-stabilized LoRA
    loftq_config = None,
    target_modules = "all-linear",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA rank: {CONFIG['lora_r']}")
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({trainable/total*100:.2f}%)")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── 4.3 Convert Dataset to Conversation Format ───────────────────────────
# Parallel image loading for speed (like Gemma4 notebook)

def convert_sample(sample):
    """Convert a JSONL sample into Unsloth vision conversation format."""
    try:
        img_path = str(Path(CONFIG["dataset_dir"]) / sample["image"])
        img = Image.open(img_path).convert("RGB")
        
        # Resize large images to save memory during training
        max_dim = 1024
        if max(img.size) > max_dim:
            ratio = max_dim / max(img.size)
            img = img.resize((int(img.width * ratio), int(img.height * ratio)), Image.LANCZOS)
        
        return {"messages": [
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": sample["conversations"][1]["content"].replace("<image>\n", "")},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["conversations"][2]["content"]},
            ]},
        ]}
    except Exception:
        return None


print(f"Converting {len(train_data)} training samples (parallel)...")
with ThreadPoolExecutor(max_workers=8) as executor:
    train_results = list(tqdm(executor.map(convert_sample, train_data),
                              total=len(train_data), desc="Train"))

train_dataset = [r for r in train_results if r is not None]
print(f"Ready: {len(train_dataset)} training conversations")
print(f"  ({len(train_data) - len(train_dataset)} skipped)")

In [ ]:
# ── 4.4 Training Configuration ────────────────────────────────────────────
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

steps_per_epoch = len(train_dataset) // (CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"])
total_steps = steps_per_epoch * CONFIG["num_epochs"]

print(f"Training plan:")
print(f"  Samples:       {len(train_dataset)}")
print(f"  Batch:         {CONFIG['batch_size']} x {CONFIG['gradient_accumulation_steps']} = {EFFECTIVE_BATCH} effective")
print(f"  Steps/epoch:   {steps_per_epoch}")
print(f"  Total steps:   {total_steps}")
print(f"  Epochs:        {CONFIG['num_epochs']}")

trainer = SFTTrainer(
    model=model,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=CONFIG["batch_size"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        num_train_epochs=CONFIG["num_epochs"],
        
        learning_rate=CONFIG["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_ratio=CONFIG["warmup_ratio"],
        weight_decay=0.01,
        max_grad_norm=1.0,
        optim="adamw_8bit",
        
        logging_steps=5,
        save_strategy="steps",
        save_steps=max(total_steps // 5, 20),  # ~5 checkpoints
        save_total_limit=3,
        
        max_seq_length=CONFIG["max_seq_length"],
        seed=3407,
        output_dir=CONFIG["output_dir"],
        report_to="none",
        
        bf16=CONFIG["bf16"],
        fp16=CONFIG["fp16"],
        
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print(f"\nTrainer ready. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── 4.5 Train! ────────────────────────────────────────────────────────────
print("Starting fine-tuning...")
print(f"  Model:    {CONFIG['model_id']}")
print(f"  LoRA:     r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']}, rsLoRA=True")
print(f"  GPU:      {gpu_name} ({vram_gb:.0f} GB)")
print(f"  Precision: {'bf16' if CONFIG['bf16'] else 'fp16'}")
print()

train_start = time.time()
train_result = trainer.train()
train_elapsed = time.time() - train_start

print(f"\n{'='*60}")
print(f"Training complete: {train_elapsed / 60:.1f} minutes")
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Peak VRAM:  {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print(f"{'='*60}")

In [ ]:
# ── 4.6 Training Loss Curve ───────────────────────────────────────────────
log_history = trainer.state.log_history

train_steps = [h["step"] for h in log_history if "loss" in h]
train_losses = [h["loss"] for h in log_history if "loss" in h]
eval_steps = [h["step"] for h in log_history if "eval_loss" in h]
eval_losses = [h["eval_loss"] for h in log_history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_losses, label="Train Loss", color="#2196F3", linewidth=2)
if eval_losses:
    ax.plot(eval_steps, eval_losses, label="Val Loss", color="#FF5722",
            linewidth=2, marker="o", markersize=6)
ax.set_xlabel("Step", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title(f"Fine-Tuning Loss — Qwen2.5-VL-3B + LoRA (r={CONFIG['lora_r']})",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/training_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 4.7 Save LoRA Adapter ────────────────────────────────────────────────
adapter_path = f"{CONFIG['output_dir']}/lora_adapter"
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)

print(f"Adapter saved to: {adapter_path}")
!du -sh {adapter_path}

---
## 5. Post-Fine-Tuning Evaluation <a id='5-post-fine-tuning-evaluation'></a>

Evaluate the fine-tuned model on the **same test set** for direct before/after comparison.

In [ ]:
# ── 5.1 Switch to Inference Mode ─────────────────────────────────────────
FastVisionModel.for_inference(model)
print("Fine-tuned model ready for evaluation.")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# ── 5.2 Run Post-Fine-Tuning Evaluation ──────────────────────────────────
finetuned_results = run_evaluation(
    model, processor, test_data,
    tag="finetuned",
    n_samples=CONFIG["eval_samples"],
)

In [ ]:
# ── 5.3 Fine-Tuned Results Summary ───────────────────────────────────────
finetuned_summary = summarize_results(finetuned_results, "FINE-TUNED (Post-Training)")

with open(f"{CONFIG['results_dir']}/finetuned_summary.json", "w") as f:
    json.dump(finetuned_summary, f, indent=2)
with open(f"{CONFIG['results_dir']}/finetuned_detailed.json", "w", encoding="utf-8") as f:
    json.dump(finetuned_results, f, ensure_ascii=False, indent=2)

---
## 6. Results Comparison & Analysis <a id='6-results-comparison--analysis'></a>

In [ ]:
# ── 6.1 Comparison Table ──────────────────────────────────────────────────
def delta(ft, bl):
    d = ft - bl
    return f"{'+'if d>0 else ''}{d:.1f}%"

comparison = [
    ["JSON Parse Rate",
     f"{baseline_summary['json_rate']:.1f}%", f"{finetuned_summary['json_rate']:.1f}%",
     delta(finetuned_summary['json_rate'], baseline_summary['json_rate'])],
    ["Header Fields (avg)",
     f"{baseline_summary['header_avg']:.1f}%", f"{finetuned_summary['header_avg']:.1f}%",
     delta(finetuned_summary['header_avg'], baseline_summary['header_avg'])],
]

for field in HEADER_FIELDS:
    bl = baseline_summary["field_scores"][field]
    ft = finetuned_summary["field_scores"][field]
    comparison.append([f"  {field}", f"{bl:.1f}%", f"{ft:.1f}%", delta(ft, bl)])

for label, key in [("Txn Count", "txn_count"), ("Txn Dates", "txn_date"),
                    ("Txn Amounts", "txn_amount"), ("Txn Descriptions", "txn_desc"),
                    ("Txn Balances", "txn_balance")]:
    bl, ft = baseline_summary[key], finetuned_summary[key]
    comparison.append([label, f"{bl:.1f}%", f"{ft:.1f}%", delta(ft, bl)])

comparison.append(["Bilingual (Hi+Gu+En)",
    f"{baseline_summary['bilingual_header']:.1f}%", f"{finetuned_summary['bilingual_header']:.1f}%",
    delta(finetuned_summary['bilingual_header'], baseline_summary['bilingual_header'])])
comparison.append(["Synthetic (En blend)",
    f"{baseline_summary['synthetic_header']:.1f}%", f"{finetuned_summary['synthetic_header']:.1f}%",
    delta(finetuned_summary['synthetic_header'], baseline_summary['synthetic_header'])])

print("\n" + "="*70)
print("  BASELINE vs FINE-TUNED — Complete Comparison")
print("="*70)
print(tabulate(comparison, headers=["Metric", "Baseline", "Fine-Tuned", "Delta"], tablefmt="simple"))

In [ ]:
# ── 6.2 Visualization: Bar Chart Comparison ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Baseline vs Fine-Tuned Performance", fontsize=16, fontweight="bold")

width = 0.35

# Plot 1: Header Fields
fields = list(baseline_summary["field_scores"].keys())
bl_s = [baseline_summary["field_scores"][f] for f in fields]
ft_s = [finetuned_summary["field_scores"][f] for f in fields]
x = np.arange(len(fields))
axes[0].bar(x - width/2, bl_s, width, label="Baseline", color="#90CAF9", edgecolor="#1565C0")
axes[0].bar(x + width/2, ft_s, width, label="Fine-Tuned", color="#A5D6A7", edgecolor="#2E7D32")
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Header Field Extraction")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f.replace("_", "\n") for f in fields], fontsize=8, rotation=45, ha="right")
axes[0].legend()
axes[0].set_ylim(0, 105)

# Plot 2: Transaction Metrics
txn_labels = ["Count", "Date", "Amount", "Desc", "Balance"]
txn_keys = ["txn_count", "txn_date", "txn_amount", "txn_desc", "txn_balance"]
txn_bl = [baseline_summary[k] for k in txn_keys]
txn_ft = [finetuned_summary[k] for k in txn_keys]
x2 = np.arange(len(txn_labels))
axes[1].bar(x2 - width/2, txn_bl, width, label="Baseline", color="#90CAF9", edgecolor="#1565C0")
axes[1].bar(x2 + width/2, txn_ft, width, label="Fine-Tuned", color="#A5D6A7", edgecolor="#2E7D32")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Transaction Extraction")
axes[1].set_xticks(x2)
axes[1].set_xticklabels(txn_labels)
axes[1].legend()
axes[1].set_ylim(0, 105)

# Plot 3: By Source
src_labels = ["Bilingual\n(Hi+Gu+En)", "Synthetic\n(En blend)", "Overall"]
src_bl = [baseline_summary["bilingual_header"], baseline_summary["synthetic_header"], baseline_summary["header_avg"]]
src_ft = [finetuned_summary["bilingual_header"], finetuned_summary["synthetic_header"], finetuned_summary["header_avg"]]
x3 = np.arange(len(src_labels))
axes[2].bar(x3 - width/2, src_bl, width, label="Baseline", color="#90CAF9", edgecolor="#1565C0")
axes[2].bar(x3 + width/2, src_ft, width, label="Fine-Tuned", color="#A5D6A7", edgecolor="#2E7D32")
axes[2].set_ylabel("Header Accuracy (%)")
axes[2].set_title("By Document Source")
axes[2].set_xticks(x3)
axes[2].set_xticklabels(src_labels)
axes[2].legend()
axes[2].set_ylim(0, 105)

plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/comparison_charts.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 6.3 Per-Sample Improvement ───────────────────────────────────────────
n_eval = min(len(baseline_results), len(finetuned_results))
bl_h = [baseline_results[i]["header_avg"] * 100 for i in range(n_eval)]
ft_h = [finetuned_results[i]["header_avg"] * 100 for i in range(n_eval)]
improvements = [ft_h[i] - bl_h[i] for i in range(n_eval)]

fig, ax = plt.subplots(figsize=(14, 6))
colors = ["#4CAF50" if d > 0 else "#F44336" if d < 0 else "#9E9E9E" for d in improvements]
ax.bar(range(n_eval), improvements, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_xlabel("Test Sample Index")
ax.set_ylabel("Header Accuracy Change (pp)")
ax.set_title("Per-Sample Improvement After Fine-Tuning", fontsize=14, fontweight="bold")

improved = sum(1 for d in improvements if d > 0)
degraded = sum(1 for d in improvements if d < 0)
ax.annotate(
    f"Improved: {improved}/{n_eval}\nDegraded: {degraded}/{n_eval}\nAvg: {np.mean(improvements):+.1f}pp",
    xy=(0.98, 0.98), xycoords="axes fraction", ha="right", va="top", fontsize=11,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray"))

plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/per_sample_improvement.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Qualitative Examples <a id='7-qualitative-examples'></a>

In [ ]:
# ── 7.1 Side-by-Side Comparison ──────────────────────────────────────────
def show_comparison(idx):
    """Show document image + field-level GT/baseline/finetuned comparison."""
    sample = test_data[idx]
    bl, ft = baseline_results[idx], finetuned_results[idx]
    gt = json.loads(sample["conversations"][2]["content"])
    
    bl_preds = json.load(open(f"{CONFIG['results_dir']}/baseline_predictions.json"))
    ft_preds = json.load(open(f"{CONFIG['results_dir']}/finetuned_predictions.json"))
    bl_json = parse_json_response(bl_preds[idx]["prediction"]) if idx < len(bl_preds) else None
    ft_json = parse_json_response(ft_preds[idx]["prediction"]) if idx < len(ft_preds) else None
    
    print(f"\n{'='*80}")
    src = 'Bilingual (Hi+Gu+En)' if sample['source'].startswith('ds1') else 'Synthetic (En blend)'
    print(f"  {sample['id']} — {src}")
    print(f"{'='*80}")
    
    img = Image.open(Path(CONFIG["dataset_dir"]) / sample["image"])
    fig, ax = plt.subplots(figsize=(8, 11))
    ax.imshow(img); ax.set_title(sample['id']); ax.axis("off")
    plt.tight_layout(); plt.show()
    
    print(f"\n{'Field':<25} {'Ground Truth':<30} {'Baseline':<30} {'Fine-Tuned':<30}")
    print("-" * 115)
    for field in HEADER_FIELDS:
        gt_v = str(gt.get(field, '—'))[:28]
        bl_v = str(bl_json.get(field, '—'))[:28] if bl_json else 'PARSE FAIL'
        ft_v = str(ft_json.get(field, '—'))[:28] if ft_json else 'PARSE FAIL'
        bl_m = '  ' if bl.get(f'field_{field}', 0) > 0.8 else '  '
        ft_m = '  ' if ft.get(f'field_{field}', 0) > 0.8 else '  '
        print(f"{field:<25} {gt_v:<30} {bl_m}{bl_v:<28} {ft_m}{ft_v:<28}")
    
    gt_n = len(gt.get('transactions', []))
    bl_n = len(bl_json.get('transactions', [])) if bl_json else 0
    ft_n = len(ft_json.get('transactions', [])) if ft_json else 0
    print(f"\n{'Txn count':<25} {gt_n:<30} {'  '}{bl_n:<28} {'  '}{ft_n:<28}")
    
    print(f"\n{'Metric':<25} {'Baseline':<15} {'Fine-Tuned':<15}")
    print("-" * 55)
    print(f"{'Header accuracy':<25} {bl['header_avg']*100:.1f}%{'':<10} {ft['header_avg']*100:.1f}%")
    print(f"{'Txn date accuracy':<25} {bl['txn_date_accuracy']*100:.1f}%{'':<10} {ft['txn_date_accuracy']*100:.1f}%")
    print(f"{'Txn amount accuracy':<25} {bl['txn_amount_accuracy']*100:.1f}%{'':<10} {ft['txn_amount_accuracy']*100:.1f}%")
    print(f"{'Inference time':<25} {bl['inference_time']:.1f}s{'':<11} {ft['inference_time']:.1f}s")


# Show 3 examples
for idx in [0, min(5, len(test_data)-1), min(10, len(test_data)-1)]:
    if idx < len(baseline_results) and idx < len(finetuned_results):
        show_comparison(idx)

In [ ]:
# ── 7.2 Indic Script Gap Analysis ────────────────────────────────────────
def contains_indic(text):
    return bool(re.search(r'[\u0900-\u097F\u0A80-\u0AFF]', text or ''))

def analyze_indic(predictions_file, tag):
    preds = json.load(open(predictions_file))
    indic_s, english_s = [], []
    
    for i, sample in enumerate(test_data[:len(preds)]):
        gt = json.loads(sample["conversations"][2]["content"])
        pred = parse_json_response(preds[i]["prediction"])
        if not pred: continue
        
        gt_txns, pred_txns = gt.get("transactions", []), pred.get("transactions", [])
        for j in range(min(len(gt_txns), len(pred_txns))):
            gt_d = gt_txns[j].get("description", "")
            pred_d = pred_txns[j].get("description", "") if isinstance(pred_txns[j], dict) else ""
            sim = fuzz.ratio(str(gt_d), str(pred_d)) / 100.0
            (indic_s if contains_indic(gt_d) else english_s).append(sim)
    
    indic_avg = np.mean(indic_s)*100 if indic_s else 0
    eng_avg = np.mean(english_s)*100 if english_s else 0
    print(f"  {tag}:")
    print(f"    Indic:   {indic_avg:.1f}% ({len(indic_s)} txns)")
    print(f"    English: {eng_avg:.1f}% ({len(english_s)} txns)")
    print(f"    Gap:     {eng_avg - indic_avg:.1f}pp")
    return {"indic": indic_avg, "english": eng_avg, "gap": eng_avg - indic_avg}

print("\nIndic Script Gap Analysis")
print("=" * 50)
bl_indic = analyze_indic(f"{CONFIG['results_dir']}/baseline_predictions.json", "Baseline")
ft_indic = analyze_indic(f"{CONFIG['results_dir']}/finetuned_predictions.json", "Fine-Tuned")
print(f"\n  Gap reduction: {bl_indic['gap'] - ft_indic['gap']:.1f}pp")

---
## 8. Export & Deployment <a id='8-export--deployment'></a>

In [ ]:
# ── 8.1 Save Final Report ────────────────────────────────────────────────
final_report = {
    "project": "Bilingual Indian Financial Document Extractor",
    "model": CONFIG["model_id"],
    "method": f"Unsloth LoRA (r={CONFIG['lora_r']}, rsLoRA=True)",
    "gpu": f"{gpu_name} ({vram_gb:.0f} GB)",
    "dataset": {
        "train": len(train_data), "val": len(val_data), "test": len(test_data),
        "sources": "1,300 synthetic Indian bank statements across DS1+DS2+DS3",
        "languages": "English, Hindi (Devanagari), Gujarati",
        "bank_formats": "12 real Indian banks (SBI/HDFC/ICICI/Axis/PNB/BOB/Kotak/Canara/UBI/BOI/IndusInd/MUCB) + 4 fictional banks across 4 layouts",
    },
    "training": {
        "epochs": CONFIG["num_epochs"],
        "effective_batch": EFFECTIVE_BATCH,
        "learning_rate": CONFIG["learning_rate"],
        "quantization": "4-bit" if CONFIG["load_4bit"] else "16-bit",
        "precision": "bf16" if CONFIG["bf16"] else "fp16",
    },
    "results": {
        "baseline": baseline_summary,
        "finetuned": finetuned_summary,
        "improvement": {
            "header_accuracy": f"{finetuned_summary['header_avg'] - baseline_summary['header_avg']:+.1f}pp",
            "json_parse_rate": f"{finetuned_summary['json_rate'] - baseline_summary['json_rate']:+.1f}pp",
            "txn_amount_accuracy": f"{finetuned_summary['txn_amount'] - baseline_summary['txn_amount']:+.1f}pp",
            "txn_description_sim": f"{finetuned_summary['txn_desc'] - baseline_summary['txn_desc']:+.1f}pp",
        },
        "indic_gap": {
            "baseline": f"{bl_indic['gap']:.1f}pp",
            "finetuned": f"{ft_indic['gap']:.1f}pp",
            "reduction": f"{bl_indic['gap'] - ft_indic['gap']:.1f}pp",
        },
    },
}

with open(f"{CONFIG['results_dir']}/final_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

print("Final report saved.")
print(json.dumps(final_report["results"]["improvement"], indent=2))

In [ ]:
# ── 8.2 Zip Artifacts for Download ───────────────────────────────────────
!cd /content && zip -r /content/bank_extractor_results.zip \
    results/ 2>/dev/null

!cd /content && zip -r /content/bank_extractor_lora.zip \
    qwen_vlm_finetuned/lora_adapter/ 2>/dev/null

for f in ['bank_extractor_results.zip', 'bank_extractor_lora.zip']:
    p = f'/content/{f}'
    if os.path.exists(p):
        print(f"{f}: {os.path.getsize(p)/1e6:.1f} MB")

print("\nTo download:")
print("  from google.colab import files")
print("  files.download('/content/bank_extractor_lora.zip')")

In [ ]:
# ── 8.3 Final Summary ────────────────────────────────────────────────────
print(f"""
{'='*64}
  Bilingual Indian Financial Document Extractor — Complete
{'='*64}

  Model:     Qwen2.5-VL-3B + Unsloth LoRA (r={CONFIG['lora_r']})
  GPU:       {gpu_name} ({vram_gb:.0f} GB)
  Dataset:   1,300 Indian bank statements, 2,913 page images
  Scripts:   English + Hindi (Devanagari) + Gujarati
  Formats:   4 layouts across 12 real + 4 fictional Indian banks

  Key Results:
    Header accuracy:  {baseline_summary['header_avg']:.1f}% -> {finetuned_summary['header_avg']:.1f}% ({finetuned_summary['header_avg'] - baseline_summary['header_avg']:+.1f}pp)
    JSON parse rate:  {baseline_summary['json_rate']:.1f}% -> {finetuned_summary['json_rate']:.1f}%
    Txn amounts:      {baseline_summary['txn_amount']:.1f}% -> {finetuned_summary['txn_amount']:.1f}%
    Indic gap:        {bl_indic['gap']:.1f}pp -> {ft_indic['gap']:.1f}pp

  Artifacts:
    results/final_report.json
    results/comparison_charts.png
    results/training_loss_curve.png
    qwen_vlm_finetuned/lora_adapter/
{'='*64}
""")

In [ ]:
# ── 8.4 Save Artifacts to Google Drive ───────────────────────────────────
# Mount Drive (skip if already mounted from dataset loading)
from google.colab import drive
import shutil

drive_mount = '/content/drive'
if not os.path.ismount(drive_mount):
    drive.mount(drive_mount)

# Create project folder on Drive
DRIVE_DEST = '/content/drive/MyDrive/indian_bank_extractor'
os.makedirs(DRIVE_DEST, exist_ok=True)

# Copy zips to Drive
artifacts = [
    '/content/bank_extractor_results.zip',
    '/content/bank_extractor_lora.zip',
]

for src in artifacts:
    if os.path.exists(src):
        dst = os.path.join(DRIVE_DEST, os.path.basename(src))
        shutil.copy2(src, dst)
        print(f"  {os.path.basename(src)} -> Drive ({os.path.getsize(dst)/1e6:.1f} MB)")
    else:
        print(f"  {os.path.basename(src)} — not found, skipping")

# Also copy the notebook itself and final report for easy access
for extra in ['/content/results/final_report.json',
              '/content/results/comparison_charts.png',
              '/content/results/training_loss_curve.png',
              '/content/results/per_sample_improvement.png']:
    if os.path.exists(extra):
        shutil.copy2(extra, DRIVE_DEST)

print(f"\nAll artifacts saved to Google Drive:")
print(f"  {DRIVE_DEST}/")
for f in sorted(os.listdir(DRIVE_DEST)):
    size = os.path.getsize(os.path.join(DRIVE_DEST, f)) / 1e6
    print(f"    {f} ({size:.1f} MB)")